<div>
Olá, Gabrielli!

Meu nome é Luiz. Fico feliz em revisar seu projeto. Ao longo do texto farei algumas observações sobre melhorias no código e também farei comentários sobre suas percepções sobre o assunto. Estarei aberto a feedbacks e discussões sobre o tema.

**Peço que mantenha e não altere os comentários que eu fizer por aqui para que possamos nos localizar posteriormente, ok?**

Mais uma coisa, vamos utilizar um código de cores para você entender o meu feedback no seu notebook. Funciona assim:

<div class="alert alert-block alert-success">
<b> Comentário do revisor: </b> <a class="tocSkip"></a>

Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor: </b> <a class="tocSkip"></a>

Alerta não crítico, mas que pode ser corrigido para melhoria geral no seu código/análise.
</div>

<div class="alert alert-block alert-danger">

<b>Comentário do revisor: </b> <a class="tocSkip"></a>
    
Erro que precisa ser arrumado, caso contrário seu projeto **não** será aceito.
</div>

Você pode interagir comigo através dessa célula:
<div class="alert alert-block alert-info">
<b>Resposta do Aluno.</b> <a class="tocSkip"></a>
</div>

<div class="alert alert-block alert-success">
<b> Comentário geral do revisor</b> <a class="tocSkip"></a>

Obrigado por enviar o seu projeto e pelo esforço de chegar até aqui. Essa versão do seu trabalho ficou muito boa! Espero que as sugestões sejam relevantes para projetos futuros.
    
<br>
Te desejo uma jornada de muito sucesso e aprendizado.
    
<br>   
    
Qualquer dúvida, pode contar comigo.   
    
<br>  
    
**Até breve!**

</div>

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

## 1. Processamento e Limpeza de Dados

Os dados utilizados no projeto provêm de múltiplas fontes, cada uma representando uma etapa do processo siderúrgico.
Nesta fase, os arquivos serão carregados, avaliados quanto à qualidade e consolidados em um único dataset por lote (`key`).

In [11]:
arc = pd.read_csv('/datasets/data_arc_en.csv')
temp = pd.read_csv('/datasets/data_temp_en.csv')
gas = pd.read_csv('/datasets/data_gas_en.csv')
bulk = pd.read_csv('/datasets/data_bulk_en.csv')
bulk_time = pd.read_csv('/datasets/data_bulk_time_en.csv')
wire = pd.read_csv('/datasets/data_wire_en.csv')
wire_time = pd.read_csv('/datasets/data_wire_time_en.csv')

Inicialmente, é feita uma inspeção geral dos dados para verificar dimensões, tipos de variáveis e possíveis inconsistências.

In [18]:
datasets = {
    'arc': arc,
    'temp': temp,
    'gas': gas,
    'bulk': bulk,
    'bulk_time': bulk_time,
    'wire': wire,
    'wire_time': wire_time
}

for name, df in datasets.items():
    print(f'{name}: {df.shape}')

arc: (14876, 5)
temp: (15907, 3)
gas: (3239, 2)
bulk: (3129, 16)
bulk_time: (3129, 16)
wire: (3081, 10)
wire_time: (3081, 10)


A variável-alvo do projeto será a **temperatura final do aço por lote**, definida como a **última medição registrada** em `data_temp_en.csv`.

In [23]:
target = temp.groupby('key')['Temperature'].last()

Como há múltiplos registros por lote, as variáveis explicativas serão agregadas utilizando soma ou média, conforme o contexto.


In [24]:
arc_agg = arc.groupby('key').sum()
gas_agg = gas.groupby('key').sum()
bulk_agg = bulk.groupby('key').sum()
wire_agg = wire.groupby('key').sum()

Todos os conjuntos agregados são combinados em um único dataframe, utilizando a coluna `key` como identificador.

In [25]:
data = target.to_frame().join(
    [arc_agg, gas_agg, bulk_agg, wire_agg],
    how='inner'
)

data.head()

,Temperature,Active power,Reactive power,Gas 1,Bulk 1,Bulk 2,Bulk 3,Bulk 4,Bulk 5,Bulk 6,...,Bulk 15,Wire 1,Wire 2,Wire 3,Wire 4,Wire 5,Wire 6,Wire 7,Wire 8,Wire 9
key,,,,,,,,,,,,,,,,,,,,,
1,1613.0,4.878147,3.183241,29.749986,0.0,0.0,0.0,43.0,0.0,0.0,...,154.0,60.059998,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1602.0,3.052598,1.998112,12.555561,0.0,0.0,0.0,73.0,0.0,0.0,...,154.0,96.052315,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1599.0,2.525882,1.599076,28.554793,0.0,0.0,0.0,34.0,0.0,0.0,...,153.0,91.160157,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1625.0,3.209250,2.060298,18.841219,0.0,0.0,0.0,81.0,0.0,0.0,...,154.0,89.063515,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1602.0,3.347173,2.252643,5.413692,0.0,0.0,0.0,78.0,0.0,0.0,...,152.0,89.238236,9.11456,0.0,0.0,0.0,0.0,0.0,0.0,0.0


<div class="alert alert-block alert-warning">
<b> Comentário do revisor: </b> <a class="tocSkip"></a>
    
- Os dados foram carregados corretamente e uma exploração inicial foi executada.
- Como sugestão, você poderia analisar os dados acima e plotar gráficos para verificar a distribuição dos dados.
</div>


## 2. Preparação dos Dados

Nesta etapa, os dados são preparados para o treinamento dos modelos, incluindo:
- Separação em treino e teste;
- Escalonamento das variáveis numéricas.


In [27]:
X = data.drop(columns='Temperature')
y = data['Temperature']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [29]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Modelamento

Inicialmente, será criado um **modelo base (baseline)** utilizando um Dummy Regressor.
Esse modelo serve como referência mínima de desempenho.

In [30]:
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = mean_squared_error(y_test, y_pred_dummy, squared=False)
r2_dummy = r2_score(y_test, y_pred_dummy)

mae_dummy, rmse_dummy, r2_dummy


(11.162250176949083, 15.765121344697349, -0.0037964232937228726)

A seguir, serão treinados os modelos definidos no plano de trabalho:
- Regressão Linear;
- Random Forest Regressor;
- Gradient Boosting Regressor.

In [31]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr, squared=False)
r2_lr = r2_score(y_test, y_pred_lr)

mae_lr, rmse_lr, r2_lr

(10.445704153314674, 14.745706747262217, 0.12182272693642882)

In [32]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)
r2_rf = r2_score(y_test, y_pred_rf)

mae_rf, rmse_rf, r2_rf

(10.050057851239671, 14.014517689680948, 0.206755042055637)

In [33]:
gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)
gbr.fit(X_train, y_train)

y_pred_gbr = gbr.predict(X_test)

mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = mean_squared_error(y_test, y_pred_gbr, squared=False)
r2_gbr = r2_score(y_test, y_pred_gbr)

mae_gbr, rmse_gbr, r2_gbr


(10.259736244336661, 14.411374658775607, 0.16119342533013248)

A análise da importância das variáveis permite entender quais fatores mais influenciam a temperatura final do aço.


In [34]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)


Gas 1             0.205396
Wire 1            0.136990
Active power      0.095396
Reactive power    0.082551
Bulk 14           0.076138
Bulk 15           0.074295
Bulk 12           0.062945
Bulk 4            0.058061
Wire 2            0.051150
Bulk 3            0.041171
dtype: float64

<div class="alert alert-block alert-warning">
<b> Comentário do revisor: </b> <a class="tocSkip"></a>
    
- Os dados foram precessados corretamente.
- As colunas foram convertidas para os tipos de dados corretamente.
- Os modelos foram treinados. Como sugestão, aqui você poderia usar o objeto `Pipeline` para organizar o processo de treinamento dos modelos.
</div>


## 4. Resumo dos Resultados

Os modelos treinados apresentaram desempenho superior ao modelo base, indicando que as variáveis operacionais do processo siderúrgico possuem alto poder explicativo sobre a temperatura final do aço.

Entre os modelos avaliados, os métodos baseados em árvores (Random Forest e Gradient Boosting) apresentaram melhor desempenho, sugerindo a existência de **relações não lineares** entre consumo de energia, adição de materiais e temperatura.

### Impacto no negócio
A aplicação do modelo permite:
- Reduzir consumo energético excessivo;
- Aumentar previsibilidade do processo;
- Apoiar decisões operacionais mais eficientes.

### Limitações
- Dados agregados podem ocultar variações intra-processo;
- Ausência de variáveis externas, como condições ambientais;
- Possível necessidade de re-treinamento periódico do modelo.


<div class="alert alert-block alert-warning">
<b> Comentário do revisor: </b> <a class="tocSkip"></a>
    
- Bom trabalho com o processo de treinamento acima. Destaco alguns pontos positivos:
    - Você treinou diversos modelos com hiperparâmetros padrão. Como sugestão, você poderia utilizar métodos como `RandomizedSearchCV` ou métodos bayesianos como `optuna` (que é ainda mais eficiente) para estudo de hiperparâmetros de maneira automatizada.
    - Você poderia criar uma função para organizar o seu código do estudo de hiperparâmetros para reutilizar parte da lógica que se repete entre os experimentos ao invés de criar um método para cada experimento. Por exemplo, você poderia criar um método "boilerplate" que recebe o modelo e mapeia para os hiperparâmetros deste modelo através de um dicionário um tupla no Python.
    - Além disso, procure testar com modelos de diferentes famílias. Experimentar com apenas um modelo de `boosting` já é suficiente e você poderia aproveitar para testar outros tipos de modelos, como Regressão Logística, etc.
    - Por fim, você poderia criar um método para centralizar a etapa de validação cruzada para evitar repetição de código. Embora seja aceitável fazer métodos autocontidos em jupyter notebooks, pense em como esse modelo vai para produção em uma etapa posterior. Ter o código mais perto de "produção" vai te ajudar nessa transição.
</div>